# Imports

In [1]:
# ------------------------------------------------------------------------------------------------------
# data_structure.py
import glob
import json
import os
import re
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, List, Optional, Tuple
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
#from post_embedder import PostEmbedder


#-------------------------------------------------------------------------------------------------------
# post_embedder.py
import torch
import re
import numpy as np
import spacy
import gensim
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification, AutoTokenizer



#-------------------------------------------------------------------------------------------------------
#dataset.py
from dataclasses import dataclass
from typing import List, Optional, Tuple
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
# from data_structure import Post, Timeline, SelfState


#-------------------------------------------------------------------------------------------------------
# train.py

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, BitsAndBytesConfig, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import pickle
from tqdm import tqdm

# from data_structure import load_all_timelines
# from dataset import PostIndex, TopKSimilarDataset
# from model import QwenSelfStatePredictor, decode_predictions, TAXONOMY_TO_INDEX


#-------------------------------------------------------------------------------------------------------
#model.py
from transformers import Qwen2Model, Qwen2PreTrainedModel, AutoTokenizer , BitsAndBytesConfig
import torch
import torch.nn as nn




/raid/home/loitongbam/anaconda3/envs/clpsych/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
# Post Embedder

In [2]:

# ── Twitter-RoBERTa task names ───────────────────────────────────────────────
_TASKS = ["emoji", "emotion", "hate", "irony", "offensive", "sentiment"]


class PostEmbedder:
    def __init__(self, wv_model_path: str, spacy_model: str = "en_core_web_sm", device: str = "cpu") -> None:
        print("[PostEmbedder] Loading Word2Vec …")
        self.wv_model = gensim.models.KeyedVectors.load_word2vec_format( wv_model_path, binary=False)
        self._wdim = self.wv_model["word"].shape[0]

        print("[PostEmbedder] Loading sentence-transformer …")
        self.sv_model = SentenceTransformer("sentence-transformers/nli-roberta-large", device=device)

        print("[PostEmbedder] Loading Twitter-RoBERTa task models …")
        self._task_models: dict[str, tuple] = {}
        for task in _TASKS:
            model_name = (f"cardiffnlp/twitter-roberta-base-{task}-latest" if task == "hate" else f"cardiffnlp/twitter-roberta-base-{task}")
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
            model.eval()
            self._task_models[task] = (model, tokenizer)

        print("[PostEmbedder] Loading spaCy …")
        self.nlp = spacy.load(spacy_model)

        self._stops = set(stopwords.words("english"))
        self._device = device
        print("[PostEmbedder] Ready.")


    def embed(self, text: str) -> np.ndarray:
        try:
            sentences = [str(s) for s in self.nlp(text).sents]
            if not sentences:
                sentences = [text]

            wv_part = self._word2vec_emb(text)
            sv_part = self._sentence_emb(sentences)
            task_part = self._task_scores(sentences)

            vec = np.concatenate([wv_part, sv_part, task_part], axis=None)

            if np.isnan(vec).any():
                raise ValueError(f"NaN values in embedding for text: {text[:60]!r}")
            return vec
        except Exception as exc:
            raise RuntimeError(f"[PostEmbedder] embed() failed: {exc}") from exc


    @staticmethod
    def _preprocess(text: str) -> str:
        """Lower-case; replace @mentions with @user; strip URLs."""
        tokens = []
        for t in text.split():
            t = t.lower()
            if t.startswith("@") and len(t) > 1:
                t = "@user"
            elif t.startswith("http"):
                t = ""
            tokens.append(t)
        return " ".join(tokens)

    def _remove_stopwords(self, text: str) -> list[str]:
        return [w for w in text.split() if w and w not in self._stops]

    def _word2vec_emb(self, text: str) -> np.ndarray:
        cleaned = self._preprocess(text)
        words = self._remove_stopwords(cleaned)
        vec = np.zeros(self._wdim)
        n = 0
        for w in words:
            if w in self.wv_model:
                vec += self.wv_model[w]
                n += 1
        if n > 0:
            vec /= n
        return vec

    def _sentence_emb(self, sentences: list[str]) -> np.ndarray:
        embeddings = self.sv_model.encode(sentences, device=self._device)
        return np.mean(embeddings, axis=0)

    def _task_score_single(self, task: str, text: str) -> np.ndarray:
        model, tokenizer = self._task_models[task]
        enc = tokenizer(text, truncation=True, max_length=512, return_tensors="pt").to(self._device)
        with torch.no_grad():
            out = model(**enc)
        return out[0][0].detach().cpu().numpy()

    def _task_scores(self, sentences: list[str]) -> np.ndarray:
        """Average Twitter-RoBERTa scores across all sentences (hate excluded from concat)."""
        per_sentence = []
        for sent in sentences:
            parts = [
                self._task_score_single(t, sent)
                for t in ["emoji", "emotion", "irony", "offensive", "sentiment" , "hate"]
            ]
            per_sentence.append(np.concatenate(parts, axis=None))
        return np.mean(per_sentence, axis=0)

---
# Data Structure
### represents the dataset as as Timeline , Post , Self-state , Subelement

In [ ]:
#data_structure.py

print("CUDA visible devices:", os.environ.get('CUDA_VISIBLE_DEVICES', 'Not Set'))
WV_MODEL_PATH  = "./wiki-news-300d-1M.vec"
device = "cuda" if torch.cuda.is_available() else "cpu"
PostEmbedder = PostEmbedder(wv_model_path=WV_MODEL_PATH, device=device)

ABCD_TAXONOMY = {
    "A": {
        "adaptive": {
            1:  "Calm/laid back",
            3:  "Sad, Emotional pain, grieving",
            5:  "Content, happy, joy, hopeful",
            7:  "Vigor/energetic",
            9:  "Justifiable anger/assertive anger, justifiable outrage",
            11: "Proud",
            13: "Feel loved, belong",
        },
        "maladaptive": {
            2:  "Anxious/fearful/tense",
            4:  "Depressed, despair, hopeless",
            6:  "Mania",
            8:  "Apathetic, don't care, blunted",
            10: "Angry (aggression), disgust, contempt",
            12: "Ashamed, guilty",
            14: "Feel lonely",
        },
    },
    "B-O": {
        "adaptive": {
            1: "Relating behavior",
            3: "Autonomous or adaptive control behavior",
        },
        "maladaptive": {
            2: "Fight or flight behavior",
            4: "Over controlled or controlling behavior",
        },
    },
    "B-S": {
        "adaptive": {
            1: "Self care and improvement",
        },
        "maladaptive": {
            2: "Self harm, neglect and avoidance",
        },
    },
    "C-O": {
        "adaptive": {
            1: "Perception of the other as related",
            3: "Perception of the other as facilitating autonomy needs",
        },
        "maladaptive": {
            2: "Perception of the other as detached or over attached",
            4: "Perception of the other as blocking autonomy needs",
        },
    },
    "C-S": {
        "adaptive": {
            1: "Self-acceptance and compassion",
        },
        "maladaptive": {
            2: "Self criticism",
        },
    },
    "D": {
        "adaptive": {
            1: "Relatedness",
            3: "Autonomy and adaptive control",
            5: "Competence, self esteem, self-care",
        },
        "maladaptive": {
            2: "Expectation that relatedness needs will not be met",
            4: "Expectation that autonomy needs will not be met",
            6: "Expectation that competence needs will not be met",
        },
    },
}


# Canonical dimension keys (as they appear in JSON)
DIMENSIONS = ["A", "B-O", "B-S", "C-O", "C-S", "D"]

# Change label constants
NO_CHANGE  = "0"
SWITCH     = "S"
ESCALATION = "E"

# Presence scale
PRESENCE_MIN = 1
PRESENCE_MAX = 5

# Date format in JSON
DATE_FORMAT = "%d-%m-%Y, %H:%M:%S"

@dataclass
class SubElement:
    dimension: str        # "A", "B-O", "B-S", "C-O", "C-S", "D"
    valence:   str        # "adaptive" | "maladaptive"
    number:    int        # e.g. 4 for "(4) Depressed, despair, hopeless"
    label:     str        # e.g. "Depressed, despair, hopeless"
    span:      str        # highlighted_evidence from post text

    @classmethod
    def from_json(cls, dimension: str, value: Dict, valence: str) -> "SubElement":
        cat_raw = value.get("Category", "")
        match = re.match(r"\((\d+)\)\s*(.+)", cat_raw)
        if match:
            number = int(match.group(1))
            label  = match.group(2).strip()
        else:
            number = 0
            label  = cat_raw.strip()

        # Fill label from taxonomy if blank
        tax_labels = ABCD_TAXONOMY.get(dimension, {}).get(valence, {})
        if number in tax_labels and not label:
            label = tax_labels[number]

        return cls(
            dimension=dimension,
            valence=valence,
            number=number,
            label=label,
            span=value.get("highlighted_evidence", "").strip(),
        )

    @property
    def short_tag(self) -> str:
        """e.g. 'A-(4)' - used in prompts and summaries."""
        return f"{self.dimension}-({self.number})"

    @property
    def full_tag(self) -> str:
        """e.g. 'A - (4) Depressed, despair, hopeless'"""
        return f"{self.dimension} - ({self.number}) {self.label}"


@dataclass
class SelfState:
    valence:     str               # "adaptive" | "maladaptive"
    subelements: List[SubElement]  # one per dimension at most (Task 1.1)
    presence:    int               # 1-5 (Task 1.2); 1 = not present

    @property
    def by_dimension(self) -> Dict[str, SubElement]:
        return {se.dimension: se for se in self.subelements}

    @property
    def dimensions_present(self) -> List[str]:
        return [se.dimension for se in self.subelements]

    @property
    def is_present(self) -> bool:
        """A self-state is considered present if presence > 1."""
        return self.presence > 1

    def to_prompt_dict(self) -> Dict:
        """Serialise back to the same JSON evidence format for prompting."""
        d = {}
        for se in self.subelements:
            d[se.dimension] = {
                "Category": f"({se.number}) {se.label}",
                "highlighted_evidence": se.span,
            }
        d["Presence"] = self.presence
        return d

def _parse_self_state(block: Dict, valence: str) -> SelfState:
    subelements = []
    presence = 1  # default: not present

    for key, value in block.items():
        if key == "Presence":
            try:
                presence = max(PRESENCE_MIN, min(PRESENCE_MAX, int(value)))
            except (TypeError, ValueError):
                presence = 1
            continue
        if not isinstance(value, dict):
            continue
        try:
            se = SubElement.from_json(key, value, valence)
            subelements.append(se)
        except Exception:
            pass

    # If no subelements found, presence must be 1 (per task spec)
    if not subelements:
        presence = 1

    return SelfState(valence=valence, subelements=subelements, presence=presence)


@dataclass
class Post:
    post_id:    str
    post_index: int
    text:       str
    timestamp:  datetime

    # Task 2: Change labels (INDEPENDENT - both can be set simultaneously)
    switch_label:     str  # "S" | "0"
    escalation_label: str  # "E" | "0"

    # Task 1.2: Well-being score (GAF-based, 1-10 or None)
    wellbeing: Optional[int]

    # Task 1.1 + 1.2: Gold self-states
    adaptive_state:    SelfState  # valence="adaptive"
    maladaptive_state: SelfState  # valence="maladaptive"

    # Whether this post has any annotation
    is_annotated: bool = False

    # Predictions (filled by pipeline)
    pred_adaptive_state:    Optional[SelfState] = None
    pred_maladaptive_state: Optional[SelfState] = None
    pred_switch_label:     str = "0"
    pred_escalation_label: str = "0"
    temporal_embedding: Optional[np.ndarray] = None
    post_embedding: Optional[np.ndarray] = None

    @property
    def is_switch(self) -> bool:
        return self.switch_label == SWITCH

    @property
    def is_escalation(self) -> bool:
        return self.escalation_label == ESCALATION

    @property
    def has_change(self) -> bool:
        return self.is_switch or self.is_escalation

    @property
    def change_tag(self) -> str:
        """Human-readable tag: 'S', 'E', 'S+E', or '-'."""
        tags = []
        if self.is_switch:     tags.append("S")
        if self.is_escalation: tags.append("E")
        return "+".join(tags) if tags else "-"

    @property
    def adaptive_presence(self) -> int:
        return self.adaptive_state.presence

    @property
    def maladaptive_presence(self) -> int:
        return self.maladaptive_state.presence

    @classmethod
    def from_dict(cls, d: Dict) -> "Post":
        try:
            ts = datetime.strptime(d["date"], DATE_FORMAT)
        except (ValueError, KeyError):
            ts = datetime.min

        switch_label     = SWITCH     if str(d.get("Switch",     "0")).upper() == "S" else "0"
        escalation_label = ESCALATION if str(d.get("Escalation", "0")).upper() == "E" else "0"

        if "well-being" in d.keys():
            wb = d.get("Well-being")
            wellbeing = int(wb) if wb is not None else None
        else:
            wellbeing = None

        if "evidence" in d.keys():
            evidence = d.get("evidence", {})
            adaptive_state    = _parse_self_state(evidence.get("adaptive-state",    {}), "adaptive")
            maladaptive_state = _parse_self_state(evidence.get("maladaptive-state", {}), "maladaptive")
            is_annotated = (
                bool(adaptive_state.subelements)
                or bool(maladaptive_state.subelements)
                or wellbeing is not None
            )
        else:
            adaptive_state    = SelfState("adaptive", [], 1)
            maladaptive_state = SelfState("maladaptive", [], 1)
            is_annotated = False

        post_embedding = PostEmbedder.embed(d.get("post", ""))

        return cls(
            post_id=d.get("post_id", ""),
            post_index=int(d.get("post_index", 0)),
            text=d.get("post", ""),
            timestamp=ts,
            switch_label=switch_label,
            escalation_label=escalation_label,
            wellbeing=wellbeing,
            adaptive_state=adaptive_state,
            maladaptive_state=maladaptive_state,
            is_annotated=is_annotated,
            post_embedding=post_embedding,
        )

@dataclass
class Timeline:
    """A complete, chronologically ordered sequence of posts for one user."""
    timeline_id: str
    posts: List[Post]

    # Stats (computed on init)
    n_posts:      int = 0
    n_annotated:  int = 0
    n_switches:   int = 0
    n_escalations: int = 0

    def __post_init__(self):
        self.posts.sort(key=lambda p: (p.timestamp, p.post_index))
        self.n_posts       = len(self.posts)
        self.n_annotated   = sum(1 for p in self.posts if p.is_annotated)
        self.n_switches    = sum(1 for p in self.posts if p.is_switch)
        self.n_escalations = sum(1 for p in self.posts if p.is_escalation)

    def hours_between(self, idx_a: int, idx_b: int) -> float:
        delta = self.posts[idx_b].timestamp - self.posts[idx_a].timestamp
        return max(0.0, delta.total_seconds() / 3600)

    def get_context(self, post_idx: int, window: int = 5) -> List[Post]:
        """Return up to `window` posts BEFORE post_idx (exclusive)."""
        start = max(0, post_idx - window)
        return self.posts[start:post_idx]

    @classmethod
    def from_dict(cls, d: Dict) -> "Timeline":
        posts = [Post.from_dict(p) for p in d.get("posts", [])]
        return cls(timeline_id=d.get("timeline_id", ""), posts=posts)



def load_timeline_file(path: str) -> Timeline:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return Timeline.from_dict(data)


def load_all_timelines(data_dir: str, pattern: str = "*.json") -> List[Timeline]:
    paths = sorted(glob.glob(os.path.join(data_dir, pattern)))
    if not paths:
        raise FileNotFoundError(f"No '{pattern}' files found in: {data_dir}")

    timelines = []
    for path in paths:
        try:
            timelines.append(load_timeline_file(path))
        except Exception as e:
            print(f"[WARNING] Skipping {path}: {e}")

    print(f"\nLoaded {len(timelines)} timelines from: {data_dir}")
    _print_dataset_stats(timelines)
    return timelines



def _print_dataset_stats(timelines: List[Timeline]) -> None:
    total_posts  = sum(tl.n_posts for tl in timelines)
    total_ann    = sum(tl.n_annotated for tl in timelines)
    total_sw     = sum(tl.n_switches for tl in timelines)
    total_esc    = sum(tl.n_escalations for tl in timelines)
    both         = sum(1 for tl in timelines
                       for p in tl.posts if p.is_switch and p.is_escalation)
    ada_subs     = sum(len(p.adaptive_state.subelements)
                       for tl in timelines for p in tl.posts)
    mal_subs     = sum(len(p.maladaptive_state.subelements)
                       for tl in timelines for p in tl.posts)

    print(f"  Timelines             : {len(timelines)}")
    print(f"  Total posts           : {total_posts}")
    print(f"  Annotated posts       : {total_ann}")
    print(f"  Switch posts          : {total_sw}")
    print(f"  Escalation posts      : {total_esc}")
    print(f"  Both (S+E) posts      : {both}")
    print(f"  Adaptive subelements  : {ada_subs}")
    print(f"  Maladaptive subelements: {mal_subs}")

---
# Dataset 


In [4]:
#dataset.py

class PostIndex:
    def __init__(self, timelines: List[Timeline], exclude_same_timeline: bool = True, skip_no_embedding: bool = True) -> None:
        self.exclude_same_timeline = exclude_same_timeline

        # Collect all (timeline_id, Post) pairs that have a valid embedding
        self._entries: List[Tuple[str, Post]] = []

        for tl in timelines:
            for post in tl.posts:
                if post.post_embedding is None:
                    if not skip_no_embedding:
                        raise ValueError(
                            f"Post {post.post_id!r} has no embedding. "
                            "Run PostEmbedder first."
                        )
                    continue
                self._entries.append((tl.timeline_id, post))

        if not self._entries:
            raise ValueError("No posts with embeddings found in the provided timelines.")

        # Stack into (N, D) float32 matrix and L2-normalise rows for cosine sim
        raw = np.stack([e[1].post_embedding for e in self._entries], axis=0).astype(np.float32)
        norms = np.linalg.norm(raw, axis=1, keepdims=True)
        norms = np.where(norms == 0, 1.0, norms)   # avoid /0 for zero vectors
        self._matrix = raw / norms                  # shape (N, D), unit vectors

        self._timeline_ids = [e[0] for e in self._entries]
        self._posts        = [e[1] for e in self._entries]

        print( f"[PostIndex] Built index: {len(self._entries)} posts, "f"embedding dim={raw.shape[1]}")

    def query( self, post: Post, query_timeline_id: str, k: int) -> Tuple[List[Post], List[float]]:
        if post.post_embedding is None:
            return [], []

        # Normalise query vector
        q = post.post_embedding.astype(np.float32)
        q_norm = np.linalg.norm(q)
        if q_norm > 0:
            q = q / q_norm

        # Cosine similarities: dot(matrix, q) because rows are already normalised
        sims = self._matrix @ q  # shape (N,)

        # Mask out: (a) the query post itself, (b) same-timeline posts if requested
        for i, (tid, p) in enumerate(self._entries):
            if p.post_id == post.post_id:
                sims[i] = -2.0   # guaranteed lowest
            elif self.exclude_same_timeline and tid == query_timeline_id:
                sims[i] = -2.0

        # Top-k indices (descending)
        top_k_idx = np.argpartition(sims, -k)[-k:]          # unsorted
        top_k_idx = top_k_idx[np.argsort(sims[top_k_idx])[::-1]]  # sorted desc

        similar_posts = [self._posts[i] for i in top_k_idx]
        scores        = [float(sims[i])  for i in top_k_idx]

        return similar_posts, scores

@dataclass
class TopKSimilarInstance:
    timeline_id:   str
    post:          Post
    similar_posts: List[Post]        # length == k (or fewer at dataset edges)
    scores:        List[float]       # parallel to similar_posts
    context_posts: List[Post]

    # ── convenience pass-throughs so code that reads Task11Instance still works
    @property
    def post_id(self)        -> str:           return self.post.post_id
    @property
    def post_index(self)     -> int:           return self.post.post_index
    @property
    def text(self)           -> str:           return self.post.text
    @property
    def adaptive_state(self) -> SelfState:     return self.post.adaptive_state
    @property
    def maladaptive_state(self) -> SelfState:  return self.post.maladaptive_state
    @property
    def wellbeing(self)      -> Optional[int]: return self.post.wellbeing
    @property
    def post_embedding(self) -> Optional[np.ndarray]: return self.post.post_embedding

    def similar_texts(self) -> List[str]:
        """Convenience: just the text of each similar post."""
        return [p.text for p in self.similar_posts]


# ── 3. Dataset ────────────────────────────────────────────────────────────────

class TopKSimilarDataset(Dataset):
    def __init__(self, timelines: List[Timeline], index: PostIndex, k: int = 5, t: int = 3, annotated_only: bool = True) -> None:
        self.k = k
        self.t = t
        self.instances: List[TopKSimilarInstance] = []

        skipped = 0
        for tl in timelines:
            for i, post in enumerate(tl.posts):
                if annotated_only and not post.is_annotated:
                    continue
                if post.post_embedding is None:
                    skipped += 1
                    continue

                similar_posts, scores = index.query(post, tl.timeline_id, k=k)
                context_posts = tl.get_context(i, window=t)

                self.instances.append(TopKSimilarInstance(
                    timeline_id=tl.timeline_id,
                    post=post,
                    similar_posts=similar_posts,
                    scores=scores,
                    context_posts=context_posts
                ))

        print(
            f"[TopKSimilarDataset] {len(self.instances)} instances built "
            f"(k={k}, skipped {skipped} posts without embedding)"
        )

    def __len__(self) -> int:
        return len(self.instances)

    def __getitem__(self, i: int) -> TopKSimilarInstance:
        return self.instances[i]

    # ── collate ───────────────────────────────────────────────────────────────

    @staticmethod
    def collate(batch: List[TopKSimilarInstance]) -> dict:
        posts         = [inst.post          for inst in batch]
        similar_posts = [inst.similar_posts for inst in batch]
        context_posts = [inst.context_posts for inst in batch]
        scores        = torch.tensor([inst.scores for inst in batch], dtype=torch.float32)


        # Labels
        ada_presence = torch.tensor([inst.post.adaptive_state.presence for inst in batch], dtype=torch.float32)
        mal_presence = torch.tensor([inst.post.maladaptive_state.presence for inst in batch], dtype=torch.float32)

        # 1. Target Post Embeddings: (B, D)
        embs = [inst.post.post_embedding for inst in batch]
        post_embeddings = torch.tensor(np.stack(embs), dtype=torch.float32) if all(e is not None for e in embs) else None

        # 2. Stack similar-post embeddings: shape (B, k, D)
        sim_embs = [[p.post_embedding for p in inst.similar_posts] for inst in batch]
        if all(e is not None for row in sim_embs for e in row):
            similar_embeddings = torch.tensor(
                np.stack([np.stack(row) for row in sim_embs]), dtype=torch.float32
            )
        else:
            similar_embeddings = None

        # 3. Temporal Context Embeddings: (B, T, D) 
        # Note: We must pad with zeros if a timeline has fewer than 't' previous posts
        ctx_embs_list = []
        max_t = max(len(inst.context_posts) for inst in batch) if batch else 0
        
        # Determine embedding dimension from first available embedding
        dim = embs[0].shape[0] if embs[0] is not None else 0

        for inst in batch:
            # Get existing embeddings
            current_ctx = [p.post_embedding for p in inst.context_posts]
            # Pad with zero vectors if the user is at the start of their timeline
            while len(current_ctx) < max_t:
                current_ctx.insert(0, np.zeros(dim)) 
            ctx_embs_list.append(np.stack(current_ctx))

        context_embeddings = torch.tensor(np.stack(ctx_embs_list), dtype=torch.float32) if dim > 0 else None
        
        return {
            "posts":              posts,
            "similar_posts":      similar_posts,
            "context_posts":      context_posts,      # List of lists
            "scores":             scores,             
            "post_embeddings":    post_embeddings,    
            "similar_embeddings": similar_embeddings, 
            "context_embeddings": context_embeddings, # (B, T, D) tensor
            "ada_presence":       ada_presence,       
            "mal_presence":       mal_presence,       
        }

---
# Model

In [5]:
#model.py
TAXONOMY_TO_INDEX = {
    "adaptive": {
        "A":   {1: 0, 3: 1, 5: 2, 7: 3, 9: 4, 11: 5, 13: 6},
        "B-O": {1: 7, 3: 8},
        "B-S": {1: 9},
        "C-O": {1: 10, 3: 11},
        "C-S": {1: 12},
        "D":   {1: 13, 3: 14, 5: 15},
    },
    "maladaptive": {
        "A":   {2: 16, 4: 17, 6: 18, 8: 19, 10: 20, 12: 21, 14: 22},
        "B-O": {2: 23, 4: 24},
        "B-S": {2: 25},
        "C-O": {2: 26, 4: 27},
        "C-S": {2: 28},
        "D":   {2: 29, 4: 30, 6: 31},
    }
}

INDEX_TO_TAXONOMY = {}
for valence, elements in TAXONOMY_TO_INDEX.items():
    for element, subelements in elements.items():
        for number, index in subelements.items():
            INDEX_TO_TAXONOMY[index] = {"valence": valence, "element": element, "number": number}

ELEMENT_SLICES = {
    "adaptive": {
        "A": (0, 7), "B-O": (7, 9), "B-S": (9, 10), 
        "C-O": (10, 12), "C-S": (12, 13), "D": (13, 16)
    },
    "maladaptive": {
        "A": (16, 23), "B-O": (23, 25), "B-S": (25, 26), 
        "C-O": (26, 28), "C-S": (28, 29), "D": (29, 32)
    }
}



class QwenSelfStatePredictor(Qwen2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.model = Qwen2Model(config)
        self.num_subelements = 32
        self.num_presence = 2
        self.classifier = nn.Linear(config.hidden_size, self.num_subelements + self.num_presence)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        
        # Get hidden state of the last non-padded token
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = input_ids.shape[0]
        last_hidden_states = outputs.last_hidden_state[torch.arange(batch_size, device=input_ids.device), sequence_lengths]
        
        logits = self.classifier(last_hidden_states)
        subelement_logits = logits[:, :self.num_subelements]
        presence_preds = logits[:, self.num_subelements:]
        
        # Loss calculation removed from forward pass
        return {"subelement_logits": subelement_logits, "presence_preds": presence_preds}

def decode_predictions(subelement_logits, presence_preds, threshold=0.5):
    """Translates tensors to the JSON dictionary format expected by CLPsych."""
    probs = torch.sigmoid(subelement_logits)
    
    ada_presence = max(1, min(5, round(presence_preds[0].item())))
    mal_presence = max(1, min(5, round(presence_preds[1].item())))
    
    prediction = {
        "adaptive-state": {"Presence": ada_presence},
        "maladaptive-state": {"Presence": mal_presence}
    }
    
    for valence in ["adaptive", "maladaptive"]:
        state_key = f"{valence}-state"
        
        for element, (start_idx, end_idx) in ELEMENT_SLICES[valence].items():
            element_probs = probs[start_idx:end_idx]
            max_prob, max_local_idx = torch.max(element_probs, dim=0)
            
            if max_prob.item() >= threshold:
                global_idx = start_idx + max_local_idx.item()
                predicted_number = INDEX_TO_TAXONOMY[global_idx]["number"]
                prediction[state_key][element] = {"subelement": predicted_number}

        # Spec constraint: if no subelements, presence MUST be 1
        if len(prediction[state_key]) == 1: # Only 'Presence' key exists
            prediction[state_key]["Presence"] = 1

    return prediction

---
# Training Script

In [ ]:
# train.py

def vectorize_target(adaptive_state, maladaptive_state):
    subelements_vec = torch.zeros(32, dtype=torch.float32)
    presence_vec = torch.zeros(2, dtype=torch.float32)
    
    if adaptive_state and hasattr(adaptive_state, 'subelements'):
        for se in adaptive_state.subelements:
            idx = TAXONOMY_TO_INDEX["adaptive"].get(se.dimension, {}).get(se.number)
            if idx is not None:
                subelements_vec[idx] = 1.0
                
    if maladaptive_state and hasattr(maladaptive_state, 'subelements'):
        for se in maladaptive_state.subelements:
            idx = TAXONOMY_TO_INDEX["maladaptive"].get(se.dimension, {}).get(se.number)
            if idx is not None:
                subelements_vec[idx] = 1.0

    presence_vec[0] = float(adaptive_state.presence) if adaptive_state else 1.0
    presence_vec[1] = float(maladaptive_state.presence) if maladaptive_state else 1.0
    
    return subelements_vec, presence_vec

def qwen_custom_collate(batch):
    batch_prompts = []
    batch_subelements = []
    batch_presence = []
    raw_posts = []

    for inst in batch:
        raw_posts.append(inst.post)
        lines = []
        # Construct Few-Shot Context

        # --- NEW: Construct Chronological Context Section ---
        lines.append("You are a clinical psychologist assistant. ")
        lines.append("Given a social media post, identify the adaptive and maladaptive and rate the presence of adaptive and maladaptive")
        lines.append("Follow the output format shown in the examples exactly.")
        lines.append("A post may contain only an adaptive self state, only a maladaptive self state, or both. Each post must have atleast one self state")
        lines.append("### Current Post History")
        for j, hist_post in enumerate(inst.context_posts):
            lines.append(f"History {j+1}")
            lines.append(f'Post: "{hist_post.text}"')
            lines.append("Output:")
            
            lines.append("  Adaptive Self-State:")
            if hist_post.adaptive_state.subelements:
                for se in hist_post.adaptive_state.subelements:
                    lines.append(f"    {se.full_tag}")
            else:
                lines.append("    none")
            lines.append(f"  Adaptive Presence: {hist_post.adaptive_state.presence} / 5")
            
            lines.append("  Maladaptive Self-State:")
            if hist_post.maladaptive_state.subelements:
                for se in hist_post.maladaptive_state.subelements:
                    lines.append(f"    {se.full_tag}")
            else:
                lines.append("    none")
            lines.append(f"  Maladaptive Presence: {hist_post.maladaptive_state.presence} / 5\n")
            lines.append("")

        lines.append("### Similar Posts to current Post")
        for rank, (ctx_post, score) in enumerate(zip(inst.similar_posts, inst.scores), 1):
            lines.append(f"### Example {rank}  (similarity: {score:.3f})")
            lines.append(f'Post: "{ctx_post.text}"')
            lines.append("Output:")
            
            lines.append("  Adaptive Self-State:")
            if ctx_post.adaptive_state.subelements:
                for se in ctx_post.adaptive_state.subelements:
                    lines.append(f"    {se.full_tag}")
            else:
                lines.append("    none")
            lines.append(f"  Adaptive Presence: {ctx_post.adaptive_state.presence} / 5")
            
            lines.append("  Maladaptive Self-State:")
            if ctx_post.maladaptive_state.subelements:
                for se in ctx_post.maladaptive_state.subelements:
                    lines.append(f"    {se.full_tag}")
            else:
                lines.append("    none")
            lines.append(f"  Maladaptive Presence: {ctx_post.maladaptive_state.presence} / 5\n")

        # Current Query Post
        lines.append("### Current Post")
        lines.append(f'Post: "{inst.text}"')
        lines.append("Output:")
        
        batch_prompts.append("\n".join(lines))

        # Vectorize Targets
        sub_vec, pres_vec = vectorize_target(inst.post.adaptive_state, inst.post.maladaptive_state)
        batch_subelements.append(sub_vec)
        batch_presence.append(pres_vec)

    return {
        "prompts": batch_prompts,
        "labels_subelements": torch.stack(batch_subelements),
        "labels_presence": torch.stack(batch_presence),
        "raw_posts": raw_posts,
        "timeline_ids": [inst.timeline_id for inst in batch]
    }


# 1. Replace Argument Parser with a Dictionary
args = {
    "model_name": "Qwen/Qwen2.5-7B",
    "train_dir": "../../data/train/",
    "eval_dir": "../../data/val/",
    "cache_dir": "./dataset_cache",
    "batch_size": 1,
    "threshold": 0.5,
    "k": 5,
    "t": 2, # Your new temporal context parameter
    "epochs": 50,
    "lr": 5e-5,
    "save_dir": "./saved_qwen_clpsych/"
}

# Update save directory logic using dictionary keys
args["save_dir"] = os.path.join(
    args["save_dir"], 
    f"{args['model_name']}_k{args['k']}_t{args['t']}_epoch{args['epochs']}"
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- 1. Load Data (with Caching) ---
os.makedirs(args["cache_dir"], exist_ok=True)
train_cache_path = os.path.join(args["cache_dir"], f"train_dataset_k{args['k']}_t{args['t']}.pkl")
eval_cache_path = os.path.join(args["cache_dir"], f"eval_dataset_k{args['k']}_t{args['t']}.pkl")

if os.path.exists(train_cache_path) and os.path.exists(eval_cache_path):
    print("Loading previously cached datasets from disk...")
    with open(train_cache_path, "rb") as f:
        train_dataset = pickle.load(f)
    with open(eval_cache_path, "rb") as f:
        eval_dataset = pickle.load(f)
else:
    print("Loading timelines and building indices...")
    train_timelines = load_all_timelines(args["train_dir"])
    eval_timelines = load_all_timelines(args["eval_dir"])
    
    train_index = PostIndex(train_timelines, exclude_same_timeline=True)
    
    print("Building Top-K datasets...")
    # Passing k and t from dictionary
    train_dataset = TopKSimilarDataset(train_timelines, train_index, k=args["k"], t=args["t"], annotated_only=True)
    eval_dataset = TopKSimilarDataset(eval_timelines, train_index, k=args["k"], t=args["t"], annotated_only=True) 
    
    with open(train_cache_path, "wb") as f:
        pickle.dump(train_dataset, f)
    with open(eval_cache_path, "wb") as f:
        pickle.dump(eval_dataset, f)
    print("Cache saved!")

train_loader = DataLoader(train_dataset, batch_size=args["batch_size"], shuffle=True, collate_fn=qwen_custom_collate)
eval_loader = DataLoader(eval_dataset, batch_size=args["batch_size"], shuffle=False, collate_fn=qwen_custom_collate)

# --- 2. Load Tokenizer & Model ---
print("Initializing Qwen Model...")
tokenizer = AutoTokenizer.from_pretrained(args["model_name"])
tokenizer.pad_token = tokenizer.eos_token 

model = QwenSelfStatePredictor.from_pretrained(
    args["model_name"], 
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_skip_modules=["classifier"]
    ), 
    device_map="auto"
)

peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    modules_to_save=["classifier"],
    lora_dropout=0.05,
)
model = get_peft_model(model, peft_config)
optimizer = torch.optim.AdamW(model.parameters(), lr=args["lr"])
bce_loss_fn = nn.BCEWithLogitsLoss()
mse_loss_fn = nn.MSELoss()

# --- 3. Training Loop ---
print("Starting Training...")
for epoch in range(args["epochs"]):
    model.train()
    total_loss = 0
    for step, batch in enumerate(train_loader):
        # Using updated max_length 2048 for temporal context
        inputs = tokenizer(batch["prompts"], padding=True, truncation=True, max_length=2048, return_tensors="pt").to(device)
        labels_subelements = batch["labels_subelements"].to(device)
        labels_presence = batch["labels_presence"].to(device)
        
        optimizer.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
            loss = bce_loss_fn(outputs["subelement_logits"].float(), labels_subelements) + (0.5 * mse_loss_fn(outputs["presence_preds"].float(), labels_presence))
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
        if step % 10 == 0:
            print(f"Epoch {epoch+1}/{args['epochs']} | Step {step} | Loss: {loss.item():.4f}")

# --- 4. Save the Model ---
os.makedirs(args["save_dir"], exist_ok=True)
model.save_pretrained(args["save_dir"])
tokenizer.save_pretrained(args["save_dir"])

# --- 5. Evaluation ---
model.eval()
submission_results = []
with torch.no_grad():
    for batch in eval_loader:
        inputs = tokenizer(batch["prompts"], padding=True, truncation=True, max_length=2048, return_tensors="pt").to(device)
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        
        sub_logits = outputs["subelement_logits"].cpu()
        pres_preds = outputs["presence_preds"].cpu()
        
        for i, (post, tid) in enumerate(zip(batch["raw_posts"], batch["timeline_ids"])):
            decoded_states = decode_predictions(sub_logits[i], pres_preds[i], threshold=args["threshold"])
            pred_obj = {
                    "timeline_id": tid,
                    "post_id": post.post_id,
                    "adaptive-state": decoded_states["adaptive-state"],
                    "maladaptive-state": decoded_states["maladaptive-state"]
                }
                
            # if len(pred_obj["adaptive-state"]) == 1 and pred_obj["adaptive-state"]["Presence"] == 1:
            #     del pred_obj["adaptive-state"]
            # if len(pred_obj["maladaptive-state"]) == 1 and pred_obj["maladaptive-state"]["Presence"] == 1:
            #     del pred_obj["maladaptive-state"]
                
            submission_results.append(pred_obj)

# Save to JSON
output_file = f"./eval_result/task1_pred_{args['model_name']}_k5_t2_epoch{args['epochs']}.json"
os.makedirs(os.path.dirname(output_file), exist_ok=True)
with open(output_file, "w") as f:
    json.dump(submission_results, f, indent=4)
    
print(f"Evaluation complete! Saved {len(submission_results)} predictions to '{output_file}'.")


Using device: cuda
Loading previously cached datasets from disk...
Initializing Qwen Model...


Loading weights: 100%|██████████| 338/338 [00:03<00:00, 109.72it/s]
QwenSelfStatePredictor LOAD REPORT from: Qwen/Qwen2.5-7B
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting Training...
Epoch 1/50 | Step 0 | Loss: 6.6930
Epoch 1/50 | Step 10 | Loss: 12.3292
Epoch 1/50 | Step 20 | Loss: 11.8112
Epoch 1/50 | Step 30 | Loss: 6.1028
Epoch 1/50 | Step 40 | Loss: 14.8161
Epoch 1/50 | Step 50 | Loss: 5.1647
Epoch 1/50 | Step 60 | Loss: 18.2341
Epoch 1/50 | Step 70 | Loss: 3.8203
Epoch 1/50 | Step 80 | Loss: 2.8489
Epoch 1/50 | Step 90 | Loss: 11.2636
Epoch 1/50 | Step 100 | Loss: 14.2609
Epoch 1/50 | Step 110 | Loss: 3.0064
Epoch 1/50 | Step 120 | Loss: 5.9608
Epoch 1/50 | Step 130 | Loss: 9.0738
Epoch 1/50 | Step 140 | Loss: 11.8485
Epoch 1/50 | Step 150 | Loss: 1.9228
Epoch 1/50 | Step 160 | Loss: 1.9316
Epoch 1/50 | Step 170 | Loss: 2.0866
Epoch 1/50 | Step 180 | Loss: 3.0009
Epoch 1/50 | Step 190 | Loss: 5.9807
Epoch 1/50 | Step 200 | Loss: 3.6965
Epoch 1/50 | Step 210 | Loss: 8.8896
Epoch 1/50 | Step 220 | Loss: 19.7433
Epoch 1/50 | Step 230 | Loss: 2.7858
Epoch 2/50 | Step 0 | Loss: 1.8798
Epoch 2/50 | Step 10 | Loss: 2.3763
Epoch 2/50 | Step 20 | 

/raid/home/loitongbam/anaconda3/envs/clpsych/lib/python3.10/site-packages/peft/utils/save_and_load.py:295: UserWarning: Could not find a config file in Qwen/Qwen2.5-7B - will assume that the vocabulary was not modified.
  warnings.warn(


Evaluation complete! Saved 21 predictions to './eval_result/task1_pred_Qwen/Qwen2.5-7B_k5_t2_epoch50.json'.


---
# Test Script

In [ ]:
# test.py

def test_collate(batch):
    """
    batch: List[TopKSimilarInstance]
    Builds prompts identical to training collate but skips target vectorisation.
    """
    prompts      = []
    raw_posts    = []
    timeline_ids = []

    for inst in batch:
        raw_posts.append(inst.post)
        timeline_ids.append(inst.timeline_id)

        lines = []
        lines.append("You are a clinical psychologist assistant. ")
        lines.append("Given a social media post, identify the adaptive and maladaptive and rate the presence of adaptive and maladaptive")
        lines.append("Follow the output format shown in the examples exactly.")
        lines.append("A post may contain only an adaptive self state, only a maladaptive self state, or both. Each post must have atleast one self state")
        lines.append("### Current Post History")
        for j, hist_post in enumerate(inst.context_posts):
            lines.append(f"History {j+1}")
            lines.append(f'Post: "{hist_post.text}"')
            lines.append("Output:")
            
            lines.append("  Adaptive Self-State:")
            if hist_post.adaptive_state.subelements:
                for se in hist_post.adaptive_state.subelements:
                    lines.append(f"    {se.full_tag}")
            else:
                lines.append("    none")
            lines.append(f"  Adaptive Presence: {hist_post.adaptive_state.presence} / 5")
            
            lines.append("  Maladaptive Self-State:")
            if hist_post.maladaptive_state.subelements:
                for se in hist_post.maladaptive_state.subelements:
                    lines.append(f"    {se.full_tag}")
            else:
                lines.append("    none")
            lines.append(f"  Maladaptive Presence: {hist_post.maladaptive_state.presence} / 5\n")
            lines.append("")

        lines.append("### Similar Posts to current Post")
        for rank, (ctx_post, score) in enumerate(zip(inst.similar_posts, inst.scores), 1):
            lines.append(f"### Example {rank}  (similarity: {score:.3f})")
            lines.append(f'Post: "{ctx_post.text}"')
            lines.append("Output:")
            
            lines.append("  Adaptive Self-State:")
            if ctx_post.adaptive_state.subelements:
                for se in ctx_post.adaptive_state.subelements:
                    lines.append(f"    {se.full_tag}")
            else:
                lines.append("    none")
            lines.append(f"  Adaptive Presence: {ctx_post.adaptive_state.presence} / 5")
            
            lines.append("  Maladaptive Self-State:")
            if ctx_post.maladaptive_state.subelements:
                for se in ctx_post.maladaptive_state.subelements:
                    lines.append(f"    {se.full_tag}")
            else:
                lines.append("    none")
            lines.append(f"  Maladaptive Presence: {ctx_post.maladaptive_state.presence} / 5\n")

        # Current Query Post
        lines.append("### Current Post")
        lines.append(f'Post: "{inst.text}"')
        lines.append("Output:")
        prompts.append("\n".join(lines))

    return {"prompts": prompts, "raw_posts": raw_posts, "timeline_ids": timeline_ids}


# 1. Replace Argument Parser with a Dictionary
args = {
    "model_name": "Qwen/Qwen2.5-7B",
    "train_dir": "../../data/train/",
    "test_dir": "../../data/test/",  # Often using val/test dir here
    "cache_dir": "./dataset_cache",
    "batch_size": 1,
    "threshold": 0.5,
    "k": 5,
    "t": 2, # New temporal context parameter
    "wv_model_path": "./wiki-news-300d-1M.vec",
    "model_weights": "./saved_qwen_clpsych/Qwen/Qwen2.5-7B_k5_t2_epoch50" # Path to your trained LoRA folder
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- 1. Load Test Data (with Caching) ---
os.makedirs(args["cache_dir"], exist_ok=True)
# Updated cache path to include 't' for consistency
test_cache_path = os.path.join(args["cache_dir"], f"test_dataset_k{args['k']}_t{args['t']}.pkl")

if os.path.exists(test_cache_path):
    print("\nLoading test dataset from cached files ...")
    with open(test_cache_path, "rb") as f:
        test_dataset = pickle.load(f)    
else:
    print("Building test dataset and index...")
    train_timelines = load_all_timelines(args["train_dir"])
    test_timelines = load_all_timelines(args["test_dir"])
    train_index = PostIndex(train_timelines, exclude_same_timeline=True)
    
    # Ensure annotated_only=False for inference
    test_dataset = TopKSimilarDataset(test_timelines, train_index, k=args["k"], t=args["t"], annotated_only=False)
    
    with open(test_cache_path, "wb") as f:
        pickle.dump(test_dataset, f)
    print(f"  Test dataset cached -> {test_cache_path}")

test_loader = DataLoader(test_dataset, batch_size=args["batch_size"], shuffle=False, collate_fn=test_collate)
print(f"  Test instances: {len(test_dataset)}")

# --- 2. Load Tokenizer & Model ---
print(f"\nLoading model from {args['model_name']} ...")
tokenizer = AutoTokenizer.from_pretrained(args["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  

# Load base model with 4-bit quantization
base_model = QwenSelfStatePredictor.from_pretrained(
    args["model_name"],
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        llm_int8_skip_modules=["classifier"],
    ),
    device_map="auto",
)

# Load the LoRA Adapters (including the trained classifier)
print(f"Loading LoRA Adapters from: {args['model_weights']}")
model = PeftModel.from_pretrained(base_model, args["model_weights"])
model.eval()

# --- 3. Running Inference ---
print("\nRunning inference ...")
submission = []
total = len(test_loader)

with torch.no_grad():
    for step, batch in enumerate(test_loader):
        # Using updated max_length 2048 for history + peer context
        # print("Batch prompts:", batch["prompts"])
        # print("Length:", len(batch["prompts"]))
        inputs = tokenizer(
            batch["prompts"],
            padding=True,
            truncation=True,
            max_length=2048, 
            return_tensors="pt",
        ).to(device)

        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        sub_logits = outputs["subelement_logits"].cpu()
        pres_preds = outputs["presence_preds"].cpu()

        for i, (post, tid) in enumerate(zip(batch["raw_posts"], batch["timeline_ids"])):
            decoded = decode_predictions(sub_logits[i], pres_preds[i], args["threshold"])

            pred_obj = {
                "timeline_id":       tid,
                "post_id":           post.post_id,
                "adaptive-state":    decoded["adaptive-state"],
                "maladaptive-state": decoded["maladaptive-state"],
            }

            # Formatting cleanup for submission
            # if (len(pred_obj["adaptive-state"]) == 1 and pred_obj["adaptive-state"]["Presence"] == 1):
            #     del pred_obj["adaptive-state"]
            # if (len(pred_obj["maladaptive-state"]) == 1 and pred_obj["maladaptive-state"]["Presence"] == 1):
            #     del pred_obj["maladaptive-state"]

            submission.append(pred_obj)

# --- 4. Write output ---
output_file = f"./test_result/task1_pred_{args['model_name']}_k5_t2_epochs50.json"
os.makedirs(os.path.dirname(os.path.abspath(output_file)), exist_ok=True)
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(submission, f, indent=4, ensure_ascii=False)

print(f"\nWrote {len(submission)} records -> {output_file}")



Using device: cuda

Loading test dataset from cached files ...
  Test instances: 92

Loading model from Qwen/Qwen2.5-7B ...


Loading weights: 100%|██████████| 338/338 [00:02<00:00, 116.21it/s]
QwenSelfStatePredictor LOAD REPORT from: Qwen/Qwen2.5-7B
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading LoRA Adapters from: ./saved_qwen_clpsych/Qwen/Qwen2.5-7B_k5_t2_epoch50

Running inference ...

Wrote 92 records -> ./test_result/task1_pred_Qwen/Qwen2.5-7B_k5_t2_epochs50.json


---
# Evaluation Script

In [ ]:
# evaluate_task1.py
"""Evaluation for Task 1.1 (ABCD subelement classification) and Task 1.2 (presence rating)."""

import argparse
import json
import sys
import warnings

import numpy as np
from scipy.stats import spearmanr
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.metrics import (
    cohen_kappa_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
)

warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

from utils import (
    ELEMENTS,
    NUM_SUBELEMENTS,
    SUBELEMENT_SCHEMA,
    VALENCES,
    build_pred_lookup,
    convert_gold_category_to_v2,
    get_gold_subelement,
    get_presence,
    get_subelement,
    load_gold_data,
    load_predictions,
    post_has_any_evidence,
    post_has_evidence,
    validate_predictions_coverage,
)


# ---------------------------------------------------------------------------
# Task 1.1 — ABCD Element Presence + Subelement Classification
# ---------------------------------------------------------------------------

def evaluate_task1_1(gold, pred_lookup, matched_keys):
    """Evaluate element presence (binary) and subelement accuracy (among TPs)."""
    results = {}

    # Element presence: binary per element x valence
    gold_elem_binary = {v: {e: [] for e in ELEMENTS} for v in VALENCES}
    pred_elem_binary = {v: {e: [] for e in ELEMENTS} for v in VALENCES}

    # Subelement: multi-class per element (0=absent, 1..K=subelement index)
    # For each post's adaptive state: vector [A_sub, BO_sub, BS_sub, CO_sub, CS_sub, D_sub]
    gold_sub_labels = {v: {e: [] for e in ELEMENTS} for v in VALENCES}
    pred_sub_labels = {v: {e: [] for e in ELEMENTS} for v in VALENCES}

    for mkey in matched_keys:
        gpost = gold[mkey]
        pred = pred_lookup[mkey]
        gevidence = gpost.get("evidence", {})

        for valence in VALENCES:
            if not post_has_evidence(gpost, valence):
                continue

            gstate = gevidence.get(valence, {})
            pstate = pred.get(valence, {})

            for elem in ELEMENTS:
                g_sub = get_gold_subelement(gstate, elem, valence)
                p_sub = get_subelement(pstate, elem)

                # Convert prediction from global (Table 1) numbering to v2
                if p_sub is not None:
                    p_sub = convert_gold_category_to_v2(
                        valence, elem, "(%d)" % p_sub)
                    # If mapping fails (invalid ID), treat as absent
                    if p_sub is None:
                        p_has = False

                g_has = g_sub is not None
                p_has = p_sub is not None

                gold_elem_binary[valence][elem].append(int(g_has))
                pred_elem_binary[valence][elem].append(int(p_has))

                # Multi-class subelement labels (0=absent)
                gold_sub_labels[valence][elem].append(g_sub if g_sub is not None else 0)
                pred_sub_labels[valence][elem].append(p_sub if p_sub is not None else 0)

    # --- Element presence metrics ---
    element_results = {}
    all_gold_elem = []
    all_pred_elem = []

    # Per-valence tracking
    valence_f1s = {v: [] for v in VALENCES}
    valence_gold = {v: [] for v in VALENCES}
    valence_pred = {v: [] for v in VALENCES}

    for valence in VALENCES:
        for elem in ELEMENTS:
            g = gold_elem_binary[valence][elem]
            p = pred_elem_binary[valence][elem]
            if not g:
                continue
            label = "%s:%s" % (valence, elem)
            prec = precision_score(g, p)
            rec = recall_score(g, p)
            f1 = f1_score(g, p)
            element_results[label] = {
                "precision": prec, "recall": rec, "f1": f1, "support": sum(g),
            }
            all_gold_elem.extend(g)
            all_pred_elem.extend(p)
            valence_f1s[valence].append(f1)
            valence_gold[valence].extend(g)
            valence_pred[valence].extend(p)

    # Overall
    elem_macro_f1 = np.mean([v["f1"] for v in element_results.values()])
    elem_micro_f1 = f1_score(all_gold_elem, all_pred_elem)

    # Per-valence aggregates
    valence_metrics = {}
    for valence in VALENCES:
        if valence_f1s[valence]:
            valence_metrics[valence] = {
                "macro_f1": np.mean(valence_f1s[valence]),
                "micro_f1": f1_score(valence_gold[valence], valence_pred[valence]),
            }

    # Average of adaptive and maladaptive
    if len(valence_metrics) == 2:
        avg_macro_f1 = np.mean([valence_metrics[v]["macro_f1"] for v in VALENCES])
        avg_micro_f1 = np.mean([valence_metrics[v]["micro_f1"] for v in VALENCES])
    else:
        avg_macro_f1 = elem_macro_f1
        avg_micro_f1 = elem_micro_f1

    results["element_presence"] = {
        "per_element": element_results,
        "adaptive_macro_f1": valence_metrics.get("adaptive-state", {}).get("macro_f1", 0.0),
        "adaptive_micro_f1": valence_metrics.get("adaptive-state", {}).get("micro_f1", 0.0),
        "maladaptive_macro_f1": valence_metrics.get("maladaptive-state", {}).get("macro_f1", 0.0),
        "maladaptive_micro_f1": valence_metrics.get("maladaptive-state", {}).get("micro_f1", 0.0),
        "avg_macro_f1": avg_macro_f1,
        "avg_micro_f1": avg_micro_f1,
        "macro_f1": elem_macro_f1,
        "micro_f1": elem_micro_f1,
    }

    # --- Subelement classification F1 (multi-class per element) ---
    # For each element: classes = {0=absent, 1, 2, ..., K}
    # Compute F1 over positive classes only (exclude 0=absent, already covered by element presence)
    sub_elem_results = {}
    all_gold_sub = []
    all_pred_sub = []

    # Per-valence tracking
    valence_sub_f1s = {v: [] for v in VALENCES}
    valence_sub_gold = {v: [] for v in VALENCES}
    valence_sub_pred = {v: [] for v in VALENCES}

    for valence in VALENCES:
        for elem in ELEMENTS:
            g = gold_sub_labels[valence][elem]
            p = pred_sub_labels[valence][elem]
            if not g:
                continue

            # Positive classes (exclude 0=absent)
            n_subs = NUM_SUBELEMENTS[valence][elem]
            pos_labels = list(range(1, n_subs + 1))

            label = "%s:%s" % (valence, elem)
            macro_f1 = f1_score(g, p, labels=pos_labels, average="macro")
            micro_f1 = f1_score(g, p, labels=pos_labels, average="micro")
            support = sum(1 for x in g if x > 0)

            sub_elem_results[label] = {
                "macro_f1": macro_f1, "micro_f1": micro_f1, "support": support,
            }
            valence_sub_f1s[valence].append(macro_f1)

            # For valence-level micro: use globally unique labels "ELEM:k"
            valence_sub_gold[valence].extend(
                ["%s:%d" % (elem, x) if x > 0 else "0" for x in g])
            valence_sub_pred[valence].extend(
                ["%s:%d" % (elem, x) if x > 0 else "0" for x in p])

    # Per-valence aggregates
    valence_sub_metrics = {}
    for valence in VALENCES:
        if valence_sub_f1s[valence]:
            g_v = valence_sub_gold[valence]
            p_v = valence_sub_pred[valence]
            pos_labels_v = sorted(set(g_v) | set(p_v))
            pos_labels_v = [l for l in pos_labels_v if l != "0"]
            valence_sub_metrics[valence] = {
                "macro_f1": np.mean(valence_sub_f1s[valence]),
                "micro_f1": f1_score(g_v, p_v, labels=pos_labels_v, average="micro"),
            }

    # Overall: pool both valences
    all_g = valence_sub_gold.get("adaptive-state", []) + valence_sub_gold.get("maladaptive-state", [])
    all_p = valence_sub_pred.get("adaptive-state", []) + valence_sub_pred.get("maladaptive-state", [])
    all_pos = sorted(set(all_g) | set(all_p))
    all_pos = [l for l in all_pos if l != "0"]
    overall_macro_f1 = np.mean([v["macro_f1"] for v in sub_elem_results.values()]) if sub_elem_results else 0.0
    overall_micro_f1 = f1_score(all_g, all_p, labels=all_pos, average="micro") if all_pos else 0.0

    # Average of adaptive and maladaptive
    if len(valence_sub_metrics) == 2:
        avg_sub_macro_f1 = np.mean([valence_sub_metrics[v]["macro_f1"] for v in VALENCES])
        avg_sub_micro_f1 = np.mean([valence_sub_metrics[v]["micro_f1"] for v in VALENCES])
    else:
        avg_sub_macro_f1 = overall_macro_f1
        avg_sub_micro_f1 = overall_micro_f1

    results["subelement_classification"] = {
        "per_element": sub_elem_results,
        "adaptive_macro_f1": valence_sub_metrics.get("adaptive-state", {}).get("macro_f1", 0.0),
        "adaptive_micro_f1": valence_sub_metrics.get("adaptive-state", {}).get("micro_f1", 0.0),
        "maladaptive_macro_f1": valence_sub_metrics.get("maladaptive-state", {}).get("macro_f1", 0.0),
        "maladaptive_micro_f1": valence_sub_metrics.get("maladaptive-state", {}).get("micro_f1", 0.0),
        "avg_macro_f1": avg_sub_macro_f1,
        "avg_micro_f1": avg_sub_micro_f1,
        "macro_f1": overall_macro_f1,
        "micro_f1": overall_micro_f1,
    }

    return results


# ---------------------------------------------------------------------------
# Task 1.2 — Presence Rating (1-5 scale)
# ---------------------------------------------------------------------------

def evaluate_task1_2(gold, pred_lookup, matched_keys):
    """Evaluate presence ratings: MAE, RMSE, QWK, Spearman."""
    ratings = {v: {"gold": [], "pred": []} for v in VALENCES}

    for key in matched_keys:
        gpost = gold[key]
        pred = pred_lookup[key]
        gevidence = gpost.get("evidence", {})

        for valence in VALENCES:
            if not post_has_evidence(gpost, valence):
                continue

            gstate = gevidence.get(valence, {})
            pstate = pred.get(valence, {})

            g_pres = get_presence(gstate)
            p_pres = get_presence(pstate)

            if g_pres is None:
                continue
            if p_pres is None:
                p_pres = 1

            ratings[valence]["gold"].append(g_pres)
            ratings[valence]["pred"].append(p_pres)

    results = {}
    all_gold = []
    all_pred = []

    for valence in VALENCES:
        g = np.array(ratings[valence]["gold"])
        p = np.array(ratings[valence]["pred"])
        if len(g) == 0:
            continue

        mae = mean_absolute_error(g, p)
        rmse = np.sqrt(mean_squared_error(g, p))
        qwk = cohen_kappa_score(g, p, weights="quadratic")
        rho, _ = spearmanr(g, p)

        results[valence] = {
            "mae": mae, "rmse": rmse, "qwk": qwk, "spearman": rho, "n": len(g),
        }
        all_gold.extend(g.tolist())
        all_pred.extend(p.tolist())

    if all_gold:
        g = np.array(all_gold)
        p = np.array(all_pred)
        results["combined"] = {
            "mae": mean_absolute_error(g, p),
            "rmse": np.sqrt(mean_squared_error(g, p)),
            "qwk": cohen_kappa_score(g, p, weights="quadratic"),
            "spearman": spearmanr(g, p)[0],
            "n": len(g),
        }

    return results


# ---------------------------------------------------------------------------
# Printing
# ---------------------------------------------------------------------------

def print_task1_1_results(results):
    print("=" * 70)
    print("TASK 1.1 — ABCD Element & Subelement Classification")
    print("=" * 70)

    ep = results["element_presence"]
    print("\n--- Element Presence (Binary) ---")
    print("%-35s %7s %7s %7s %6s" % ("Element", "Prec", "Rec", "F1", "Supp"))
    print("-" * 65)
    for label, m in sorted(ep["per_element"].items()):
        print("%-35s %7.3f %7.3f %7.3f %6d" % (
            label, m["precision"], m["recall"], m["f1"], m["support"]))
    print("-" * 65)
    print("%-35s %7s %7s %7.3f" % ("Adaptive Macro F1", "", "", ep["adaptive_macro_f1"]))
    print("%-35s %7s %7s %7.3f" % ("Adaptive Micro F1", "", "", ep["adaptive_micro_f1"]))
    print("%-35s %7s %7s %7.3f" % ("Maladaptive Macro F1", "", "", ep["maladaptive_macro_f1"]))
    print("%-35s %7s %7s %7.3f" % ("Maladaptive Micro F1", "", "", ep["maladaptive_micro_f1"]))
    print("%-35s %7s %7s %7.3f" % ("Avg Macro F1", "", "", ep["avg_macro_f1"]))
    print("%-35s %7s %7s %7.3f" % ("Avg Micro F1", "", "", ep["avg_micro_f1"]))
    print("%-35s %7s %7s %7.3f" % ("Overall Macro F1", "", "", ep["macro_f1"]))
    print("%-35s %7s %7s %7.3f" % ("Overall Micro F1", "", "", ep["micro_f1"]))

    sc = results["subelement_classification"]
    print("\n--- Subelement Classification (multi-class per element) ---")
    print("%-35s %7s %7s %6s" % ("Element", "MacroF1", "MicroF1", "Supp"))
    print("-" * 55)
    for label, m in sorted(sc["per_element"].items()):
        print("%-35s %7.3f %7.3f %6d" % (
            label, m["macro_f1"], m["micro_f1"], m["support"]))
    print("-" * 55)
    print("%-35s %7.3f %7.3f" % ("Adaptive", sc["adaptive_macro_f1"], sc["adaptive_micro_f1"]))
    print("%-35s %7.3f %7.3f" % ("Maladaptive", sc["maladaptive_macro_f1"], sc["maladaptive_micro_f1"]))
    print("%-35s %7.3f %7.3f" % ("Avg", sc["avg_macro_f1"], sc["avg_micro_f1"]))
    print("%-35s %7.3f %7.3f" % ("Overall", sc["macro_f1"], sc["micro_f1"]))


def print_task1_2_results(results):
    print("\n" + "=" * 70)
    print("TASK 1.2 — Presence Rating (1-5)")
    print("=" * 70)
    print("%-25s %7s %7s %7s %7s %6s" % ("Valence", "MAE", "RMSE", "QWK", "Spear", "N"))
    print("-" * 62)
    for label in VALENCES + ["combined"]:
        if label not in results:
            continue
        m = results[label]
        print("%-25s %7.3f %7.3f %7.3f %7.3f %6d" % (
            label, m["mae"], m["rmse"], m["qwk"], m["spearman"], m["n"]))


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def evaluate(gold_dir, pred_file):
    """Run full Task 1 evaluation. Returns results dict."""
    gold = load_gold_data(gold_dir)
    predictions = load_predictions(pred_file)
    pred_lookup = build_pred_lookup(predictions)

    matched = validate_predictions_coverage(
        gold, pred_lookup, "Task 1",
        filter_fn=post_has_any_evidence,
    )

    results = {
        "task1_1": evaluate_task1_1(gold, pred_lookup, matched),
        "task1_2": evaluate_task1_2(gold, pred_lookup, matched),
    }

    print_task1_1_results(results["task1_1"])
    print_task1_2_results(results["task1_2"])

    return results


def main():
    parser = argparse.ArgumentParser(description="Evaluate Task 1 (v4)")
    parser.add_argument("--gold-dir", required=True, help="Directory with gold JSON timeline files")
    parser.add_argument("--pred-file", required=True, help="Prediction JSON file")
    parser.add_argument("--output", help="Optional: save results to JSON file")
    args = parser.parse_args()

    results = evaluate(args.gold_dir, args.pred_file)

    if args.output:
        def convert(obj):
            if isinstance(obj, (np.integer,)):
                return int(obj)
            if isinstance(obj, (np.floating,)):
                return float(obj)
            if isinstance(obj, np.ndarray):
                return obj.tolist()
            return obj

        with open(args.output, "w") as f:
            json.dump(results, f, indent=2, default=convert)
        print("\nResults saved to %s" % args.output)


if __name__ == "__main__":
    main()

In [ ]:
#utils.py
"""Shared utilities for CLPsych 2026 evaluation scripts."""

import json
import re
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple

# The 6 ABCD elements
ELEMENTS = ["A", "B-O", "B-S", "C-O", "C-S", "D"]

# Two valences
VALENCES = ["adaptive-state", "maladaptive-state"]

# ---------------------------------------------------------------------------
# Subelement schema: {valence: {element: {index: name}}}
# Indices are 1-based, per valence per element.
# ---------------------------------------------------------------------------

SUBELEMENT_SCHEMA = {
    "adaptive-state": {
        "A": {
            1: "Calm (laid back)",
            2: "Sad (emotional pain, grieving)",
            3: "Happy (content, joyful, hopeful)",
            4: "Vigor (energy)",
            5: "Justifiably angry (assertive anger)",
            6: "Proud",
            7: "Feeling loved",
        },
        "B-O": {
            1: "Relating behavior",
            2: "Autonomous behavior",
        },
        "B-S": {
            1: "Self-care",
        },
        "C-O": {
            1: "Related",
            2: "Facilitating autonomy",
        },
        "C-S": {
            1: "Self-acceptance",
        },
        "D": {
            1: "Relatedness",
            2: "Autonomy",
            3: "Competence",
        },
    },
    "maladaptive-state": {
        "A": {
            1: "Anxious (fearful, tense)",
            2: "Depressed (despair, hopeless)",
            3: "Mania",
            4: "Apathetic (blunted affect)",
            5: "Angry (aggression, disgust, contempt)",
            6: "Ashamed (guilty)",
            7: "Loneliness",
        },
        "B-O": {
            1: "Fight or flight",
            2: "Overcontrolled",
        },
        "B-S": {
            1: "Self-harm",
        },
        "C-O": {
            1: "Detached or over-attached",
            2: "Blocking autonomy",
        },
        "C-S": {
            1: "Self-criticism",
        },
        "D": {
            1: "Relatedness unmet",
            2: "Autonomy unmet",
            3: "Competence unmet",
        },
    },
}

# Number of subelements per element x valence (for validation & evaluation)
NUM_SUBELEMENTS = {
    valence: {elem: len(subs) for elem, subs in elems.items()}
    for valence, elems in SUBELEMENT_SCHEMA.items()
}

# ---------------------------------------------------------------------------
# Gold data conversion: old global category number -> v2 index
# The old format uses global numbering across both valences per element.
# ---------------------------------------------------------------------------

GOLD_CATEGORY_MAP = {
    "adaptive-state": {
        "A":   {1: 1, 3: 2, 5: 3, 7: 4, 9: 5, 11: 6, 13: 7},
        "B-O": {1: 1, 3: 2},
        "B-S": {1: 1},
        "C-O": {1: 1, 3: 2},
        "C-S": {1: 1},
        "D":   {1: 1, 3: 2, 5: 3},
    },
    "maladaptive-state": {
        "A":   {2: 1, 4: 2, 6: 3, 8: 4, 10: 5, 12: 6, 14: 7},
        "B-O": {2: 1, 4: 2},
        "B-S": {2: 1},
        "C-O": {2: 1, 4: 2},
        "C-S": {2: 1},
        "D":   {2: 1, 4: 2, 6: 3},
    },
}

# Cross-annotations: subelements that appear in the "wrong" valence in gold data.
# Map them to the correct v2 index within that valence context.
# E.g., (2) Anxious appearing in adaptive-state:A -> treat as adaptive index 0 (skip)
# We handle these by also including cross-valence mappings.
GOLD_CATEGORY_MAP_CROSS = {
    "adaptive-state": {
        "A":   {2: None},       # Anxious is maladaptive, skip in adaptive context
        "B-O": {2: None},       # Fight/flight is maladaptive
        "D":   {4: None},       # Autonomy unmet is maladaptive
    },
    "maladaptive-state": {
        "A":   {9: None},       # Justifiable anger is adaptive
        "B-O": {1: None},       # Relating is adaptive
    },
}


def extract_category_number(category_str):
    """Extract the number from a gold category string like '(1) Calm/ laid back'."""
    if not category_str:
        return None
    m = re.search(r"\((\d+)\)", str(category_str))
    return int(m.group(1)) if m else None


def convert_gold_category_to_v2(valence, element, category_str):
    """Convert a gold Category string to a v2 subelement index.

    Returns the v2 index (int) or None if the category is a cross-annotation
    or cannot be mapped.
    """
    old_num = extract_category_number(category_str)
    if old_num is None:
        return None

    # Try primary mapping
    mapping = GOLD_CATEGORY_MAP.get(valence, {}).get(element, {})
    if old_num in mapping:
        return mapping[old_num]

    # Check cross-annotation mapping
    cross = GOLD_CATEGORY_MAP_CROSS.get(valence, {}).get(element, {})
    if old_num in cross:
        return cross[old_num]  # None = skip

    return None


def get_presence(state):
    """Get the Presence value from a state dict, returning None if absent or invalid."""
    pres = state.get("Presence")
    if pres is None or isinstance(pres, dict):
        return None
    try:
        return int(pres)
    except (ValueError, TypeError):
        return None


def post_has_evidence(post, valence):
    """Check whether a gold post has valid evidence for a given valence."""
    evidence = post.get("evidence", {})
    state = evidence.get(valence, {})
    return get_presence(state) is not None


def post_has_any_evidence(post):
    """Check whether a gold post has evidence for at least one valence."""
    return any(post_has_evidence(post, v) for v in VALENCES)


def convert_gold_post_to_v2(post):
    """Convert a gold post's evidence to v2 format (subelement index).

    Returns a dict in the v2 prediction format.
    """
    evidence = post.get("evidence", {})
    result = {
        "timeline_id": post.get("timeline_id", ""),
        "post_id": post["post_id"],
    }

    for valence in VALENCES:
        state = evidence.get(valence, {})
        pres = get_presence(state)
        if pres is None:
            continue

        v2_state = {"Presence": pres}
        for elem in ELEMENTS:
            if elem not in state or "Category" not in state[elem]:
                continue
            idx = convert_gold_category_to_v2(valence, elem, state[elem]["Category"])
            if idx is not None:
                v2_state[elem] = {"subelement": idx}

        result[valence] = v2_state

    return result


def load_gold_data(gold_dir):
    """Load gold timeline JSON files from a directory.

    Returns:
        dict mapping (timeline_id, post_id) -> post dict (original format, with timeline_id added)
    """
    gold = {}
    gold_path = Path(gold_dir)
    for fpath in sorted(gold_path.glob("*.json")):
        with open(fpath) as f:
            timeline = json.load(f)
        tid = timeline["timeline_id"]
        for post in timeline["posts"]:
            post["timeline_id"] = tid
            pid = post["post_id"]
            gold[(tid, pid)] = post
    return gold


def load_predictions(pred_file):
    """Load a prediction JSON file (list of per-post predictions)."""
    with open(pred_file) as f:
        return json.load(f)


def build_pred_lookup(predictions):
    """Build a lookup from (timeline_id, post_id) -> prediction dict."""
    lookup = {}
    for pred in predictions:
        key = (pred["timeline_id"], pred["post_id"])
        lookup[key] = pred
    return lookup


def get_subelement(state, element):
    """Extract the subelement index from a v2 state dict for an element.

    Returns an int or None if element is absent.
    """
    if element not in state:
        return None
    elem_data = state[element]
    if isinstance(elem_data, dict) and "subelement" in elem_data:
        return elem_data["subelement"]
    return None


def get_gold_subelement(state, element, valence):
    """Extract subelement index from GOLD state dict (old format with Category).

    Converts the old Category number to v2 index.
    """
    if element not in state or "Category" not in state[element]:
        return None
    return convert_gold_category_to_v2(valence, element, state[element]["Category"])


def validate_predictions_coverage(gold, pred_lookup, task_name, filter_fn=None):
    """Check that predictions cover all required gold posts.

    Returns list of (timeline_id, post_id) keys that are in both gold and pred.
    """
    missing = []
    matched = []
    for key, post in gold.items():
        if filter_fn and not filter_fn(post):
            continue
        if key not in pred_lookup:
            missing.append(key)
        else:
            matched.append(key)

    if missing:
        print("[%s] WARNING: %d gold posts missing from predictions:" % (task_name, len(missing)))
        for tid, pid in missing[:10]:
            print("  timeline=%s, post=%s" % (tid, pid))
        if len(missing) > 10:
            print("  ... and %d more" % (len(missing) - 10))

    return matched